In [95]:
# Code to map Maven elements to Origami's Treatment elements
# Using semantic similarity library
# Author: Sandeep Chintabathina
# May 2026

In [96]:
# For each Maven data element, check if there is an exact or close matching data element in the origami Treatment package
# The output will list the maven element alongside the origami element and other supporting attributes

In [97]:
from sentence_transformers import SentenceTransformer


In [98]:
import pandas as pd
import numpy as np

In [99]:
# Read maven elements
df_maven = pd.read_csv('output/all_db_elements_use_this.csv',na_filter=False)
len(df_maven)

6232

In [100]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6232 entries, 0 to 6231
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   QUESTION_ID     6232 non-null   object
 1   value           6232 non-null   object
 2   QUESTIONSET_ID  6232 non-null   object
 3   MODEL_NAME      6232 non-null   object
dtypes: object(4)
memory usage: 194.9+ KB


In [101]:
def clean_up(name):
    new_name =''
    for ch in name:
        # If space, digits, uppercase letters, lowercase letters, or underscore, add it to new name otherwise ignore
        if (ord(ch)==32 or 48<=ord(ch)<=57 or 65<=ord(ch)<=90 or 97<=ord(ch)<=122 or ord(ch)==95 ):
            new_name+=ch
    return new_name.strip()

In [102]:
clean_up('[sdfs] 34_AS ^#~')

'sdfs 34_AS'

In [103]:
# Clean up questions or data elements
df_maven.loc[:,'QUESTION_ID'] =[ clean_up(q) for q in df_maven.loc[:,'QUESTION_ID']]

In [104]:
df_maven.loc[:,'MODEL_NAME'].value_counts(dropna=False)

MODEL_NAME
DiseaseSurveillanceModel_General        3192
DiseaseSurveillanceModel_STD            1361
DiseaseSurveillanceModel_Hep             754
Childhood_Lead_Child_Model               386
ClusterModel                             205
Childhood_Lead_Investigation_Model       136
DiseaseSurveillanceModel_Coinfection     114
PortalApplicationModel                    41
EHRAndScreeningModel_STD                  22
DiseaseSurveillanceModel_AggNETSS         14
IsolationAndControlModel                   7
Name: count, dtype: int64

In [105]:
# General clean up
df_maven.loc[:,'MODEL_NAME'] = [m.strip() for m in df_maven.loc[:,'MODEL_NAME']]

In [106]:
# creating a sorting order for model name which is then used to sort the dataframe 
# Using the order from value counts above
sorter = ['DiseaseSurveillanceModel_General','DiseaseSurveillanceModel_STD','DiseaseSurveillanceModel_Hep','Childhood_Lead_Child_Model',
          'ClusterModel','Childhood_Lead_Investigation_Model','DiseaseSurveillanceModel_Coinfection','PortalApplicationModel',
          'EHRAndScreeningModel_STD','DiseaseSurveillanceModel_AggNETSS','IsolationAndControlModel']

In [107]:
# convert model_name column to category
df_maven.MODEL_NAME = df_maven.MODEL_NAME.astype('category')
# Set sorter as the categories hierarchy
df_maven.MODEL_NAME = df_maven.MODEL_NAME.cat.set_categories(sorter)


In [108]:
df_maven.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6232 entries, 0 to 6231
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   QUESTION_ID     6232 non-null   object  
 1   value           6232 non-null   object  
 2   QUESTIONSET_ID  6232 non-null   object  
 3   MODEL_NAME      6232 non-null   category
dtypes: category(1), object(3)
memory usage: 152.7+ KB


In [109]:
# Sort df on model_name
df_maven = df_maven.sort_values(['MODEL_NAME'])

In [110]:
df_maven = df_maven.reset_index(drop=True)

In [111]:
df_maven

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
0,CURRENT_TELEPHONE_WORK,(999) 999-9999,DEMOGRAPHIC,DiseaseSurveillanceModel_General
1,EAT_YOGURT_PURCHASED_FROM,Yogurt Melts,RISK_HISTORY,DiseaseSurveillanceModel_General
2,CDR_PERSON_OR_AGENCY_REPORTING,Zyan Paresa,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_General
3,ANTACID_EXPOSURE,UNKNOWN,CLINICAL,DiseaseSurveillanceModel_General
4,DIARY_DAY_1_DINNER,Zippys -No Bean Chili,RISK_HISTORY,DiseaseSurveillanceModel_General
...,...,...,...,...
6227,PROGRAM_CODE,STD,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6228,PRODUCT_DESCRIPTION,YERSINIOSIS; Yersinia enterocolitica; psudotob...,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6229,REPORT_CATEGORY_LAB,Urgent - upon order,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6230,REPORT_CATEGORY_PROVIDER,Urgent,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel


In [112]:
# duplicates
df_maven[df_maven.duplicated(['QUESTION_ID'],keep=False)]

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
0,CURRENT_TELEPHONE_WORK,(999) 999-9999,DEMOGRAPHIC,DiseaseSurveillanceModel_General
2,CDR_PERSON_OR_AGENCY_REPORTING,Zyan Paresa,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_General
10,CASE_REFERRED,YES,ADMINISTRATIVE,DiseaseSurveillanceModel_General
12,PRODUCT_DESCRIPTION,YERSINIOSIS; Yersinia enterocolitica; psudotob...,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_General
17,REPORTING_SOURCE_TYPE,SELF,ADMINISTRATIVE,DiseaseSurveillanceModel_General
...,...,...,...,...
6222,EPISODE_DATE,12/31/2016,ADMINISTRATIVE,DiseaseSurveillanceModel_AggNETSS
6224,MMWR_WEEK,53,ADMINISTRATIVE,DiseaseSurveillanceModel_AggNETSS
6226,ISO_QUAR_CONTACT_INSTRUCTIONS,"Surveillance, testing, counseling, interview, ...",ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
6228,PRODUCT_DESCRIPTION,YERSINIOSIS; Yersinia enterocolitica; psudotob...,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel


In [113]:
# Keeping the first appearance of question
df_maven = df_maven.drop_duplicates(['QUESTION_ID'])

In [114]:
df_maven[df_maven.duplicated(['QUESTION_ID'],keep=False)]

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME


In [115]:
df_maven = df_maven.reset_index(drop=True)

In [116]:
df_maven

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
0,CURRENT_TELEPHONE_WORK,(999) 999-9999,DEMOGRAPHIC,DiseaseSurveillanceModel_General
1,EAT_YOGURT_PURCHASED_FROM,Yogurt Melts,RISK_HISTORY,DiseaseSurveillanceModel_General
2,CDR_PERSON_OR_AGENCY_REPORTING,Zyan Paresa,PORTAL_CDR_CASE_SUBMISSION,DiseaseSurveillanceModel_General
3,ANTACID_EXPOSURE,UNKNOWN,CLINICAL,DiseaseSurveillanceModel_General
4,DIARY_DAY_1_DINNER,Zippys -No Bean Chili,RISK_HISTORY,DiseaseSurveillanceModel_General
...,...,...,...,...
5439,COMMENTS,whatever,ADMINISTRATIVE,DiseaseSurveillanceModel_AggNETSS
5440,PRODUCT_CODE,YER,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
5441,PROGRAM_CODE,STD,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel
5442,REPORT_CATEGORY_LAB,Urgent - upon order,ISOLATION_AND_CONTROL_INSTRUCTIONS,IsolationAndControlModel


In [117]:
# HealthcareUtilization elements
df_tr = pd.read_csv('origami/csv/Custom__Treatment.csv',na_filter=False)
len(df_tr)

312

In [118]:
df_tr.loc[:,'Label'].value_counts(dropna=False)

Label
                                 234
Medication Other                   2
Duration Units                     2
Med Route Other                    2
Dose Units                         2
                                ... 
Treatment End Date                 1
Custom Multi Select Code10 ID      1
Custom Multi Select Code1 ID       1
Client                             1
Avian Medication                   1
Name: count, Length: 73, dtype: int64

In [119]:
# Get the two columns of interest and non-blank values
df_tr = df_tr[['Field','Label']]
df_tr = df_tr[df_tr['Label']!='']
len(df_tr)

78

In [120]:
df_tr = df_tr.reset_index(drop=True)

In [121]:
df_tr

,Field,Label
0,TreatmentID,Treatment ID
1,ClientID,Client
2,EntryUserID,Entry User
3,EntryDate,Entry Date
4,ModifiedUserID,Modified User
...,...,...
73,CustomCode30ID,Frequency Unit
74,CustomCode31ID,Duration Units
75,CustomCode32ID,Reason for medication
76,CustomCode33ID,UBOLA_IV_FLUIDS


In [122]:
# Original label to be used later for output
df_tr.rename(columns={'Label':'original_label'},inplace=True)
df_tr.columns

Index(['Field', 'original_label'], dtype='object')

In [123]:
# Remove only the first occurrence of [] in a string
def remove_squares(somestring):
    if ']' in somestring:
        for i in range(len(somestring)):
            if somestring[i]==']':
                return somestring[i+1:].strip()
    return somestring.strip()


In [124]:
remove_squares('asd[asdasd]asc[obas]da')

'asc[obas]da'

In [125]:
# Process the column values (eliminate the values in square brackets)
df_tr.loc[:,'Label'] = [ remove_squares(label) for label in df_tr.loc[:,'original_label']]

In [126]:
# Clean up column names in both data sets
df_tr.loc[:,'Label'] =[ clean_up(q) for q in df_tr.loc[:,'Label']]

In [127]:
df_tr.loc[:,'Label'].value_counts()

Label
Duration Units                   2
Dose                             2
Dose Units                       2
Med Route Other                  2
Medication Other                 2
                                ..
Treatment End Date               1
Custom Multi Select Code10 ID    1
Custom Multi Select Code1 ID     1
Custom Multi Select Code2 ID     1
Avian Medication                 1
Name: count, Length: 72, dtype: int64

In [128]:
# duplicates labels exist 
df_tr[df_tr.duplicated(['Label'],keep=False)]

,Field,original_label,Label
16,CustomCode1ID,Dose Units,Dose Units
18,CustomCode2ID,Duration Units,Duration Units
28,CustomDate3,Date Treatment Prescribed,Date Treatment Prescribed
30,CustomDate5,Date Treatment Prescribed,Date Treatment Prescribed
44,CustomNumber1,Dose,Dose
47,CustomNumber4,Dose,Dose
54,CustomText2,Med Route Other,Med Route Other
57,CustomText5,Medication Other,Medication Other
58,CustomText6,Medication Other,Medication Other
59,CustomText7,Med Route Other,Med Route Other


In [129]:
# dropping them
df_tr = df_tr.drop_duplicates(['Label'])

In [130]:
df_tr[df_tr.duplicated(['Label'],keep=False)]

,Field,original_label,Label


In [131]:
df_tr = df_tr.reset_index(drop=True)
df_tr

,Field,original_label,Label
0,TreatmentID,Treatment ID,Treatment ID
1,ClientID,Client,Client
2,EntryUserID,Entry User,Entry User
3,EntryDate,Entry Date,Entry Date
4,ModifiedUserID,Modified User,Modified User
...,...,...,...
67,CustomCode29ID,Medication Route,Medication Route
68,CustomCode30ID,Frequency Unit,Frequency Unit
69,CustomCode32ID,Reason for medication,Reason for medication
70,CustomCode33ID,UBOLA_IV_FLUIDS,UBOLA_IV_FLUIDS


In [132]:
# Do a semantic match between Maven elements and Origami HAI labels

In [133]:
maven_elements = [str.replace(maven_de,'_',' ') for maven_de in df_maven.QUESTION_ID]
origami_elements = [str.replace(col,'_',' ') for col in df_tr.Label]

In [134]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [135]:
# Compute embeddings
embeddings_maven = model.encode(maven_elements)
embeddings_origami = model.encode(origami_elements)

In [136]:
print(len(embeddings_origami))
print(len(embeddings_maven))
embeddings_origami
embeddings_maven

72
5444


array([[-0.16024935,  0.0271387 ,  0.06354488, ..., -0.00712903,
        -0.01673306,  0.06124594],
       [ 0.04245438,  0.03854417,  0.0684232 , ..., -0.06141916,
         0.08061311, -0.07027474],
       [-0.08168999,  0.03473966, -0.06668424, ..., -0.04572672,
        -0.00667135,  0.0398071 ],
       ...,
       [-0.06123391,  0.04124329, -0.09608547, ...,  0.11074385,
         0.03158536, -0.0520657 ],
       [ 0.03361172,  0.06674883, -0.07311527, ...,  0.01481686,
         0.04655252, -0.02631201],
       [-0.00503135,  0.02209355, -0.08806273, ..., -0.04621199,
         0.04993448,  0.04251292]], dtype=float32)

In [137]:
# Compute cosine similarities
similarities = model.similarity(embeddings_maven,embeddings_origami)

In [138]:
similarities

tensor([[ 0.0353,  0.1604,  0.0948,  ..., -0.0374,  0.0097, -0.0180],
        [ 0.0371,  0.0250,  0.0455,  ...,  0.0416,  0.0368,  0.1673],
        [ 0.2163,  0.2240,  0.2282,  ..., -0.0038, -0.0491, -0.0550],
        ...,
        [ 0.1461,  0.2358,  0.2094,  ...,  0.0909, -0.0256,  0.1376],
        [ 0.1757,  0.0612,  0.0157,  ...,  0.0527,  0.1127,  0.0335],
        [ 0.1413,  0.1426,  0.0463,  ...,  0.0323, -0.0439,  0.0287]])

In [139]:
print(type(similarities))
len(similarities)

<class 'torch.Tensor'>


5444

In [140]:
df_similarities = pd.DataFrame(similarities,index=df_maven.QUESTION_ID,columns=df_tr.original_label)

In [141]:
df_similarities.to_csv('output/similarity_scores_Treatment.csv')

In [142]:
df_maven = df_maven.reset_index(drop=True)
df_maven.columns

Index(['QUESTION_ID', 'value', 'QUESTIONSET_ID', 'MODEL_NAME'], dtype='object')

In [143]:
df_output = df_maven[['MODEL_NAME','QUESTION_ID','QUESTIONSET_ID']]

In [144]:
#df_similarities.info()

In [145]:
#For every maven element (index), identify origami elements (column) that is the best match

filtered=[]
for idx in df_similarities.index:
    # Looking for columns with score higher than 0.75
    v = df_similarities.loc[idx,:]>0.75

    # Get the column with the maximum score
    max=0.0
    max_col=''
    for i,col in enumerate(df_similarities.columns):
        if v.iloc[i]==True:
            #print('value',df_similarities.loc[idx,col])
            if (df_similarities.loc[idx,col] > max):
                max = df_similarities.loc[idx,col]
                max_col=col

    #filtered.append({col:df_similarities.loc[idx,col]})
    #print(idx+','+max_col+':'+str(max)+'\n')
    filtered.append({'QUESTION_ID':idx,'target_label':max_col})



In [146]:
df_filtered = pd.DataFrame(filtered)
print(len(df_filtered))
#df_filtered.loc[:,'target_label'].value_counts(dropna=False)

5444


In [147]:
df_merge = pd.merge(df_filtered,df_tr,left_on='target_label',right_on='original_label',how='left')

In [148]:
df_merge

,QUESTION_ID,target_label,Field,original_label,Label
0,CURRENT_TELEPHONE_WORK,,NaN,NaN,NaN
1,EAT_YOGURT_PURCHASED_FROM,,NaN,NaN,NaN
2,CDR_PERSON_OR_AGENCY_REPORTING,,NaN,NaN,NaN
3,ANTACID_EXPOSURE,,NaN,NaN,NaN
4,DIARY_DAY_1_DINNER,,NaN,NaN,NaN
...,...,...,...,...,...
5439,COMMENTS,,NaN,NaN,NaN
5440,PRODUCT_CODE,,NaN,NaN,NaN
5441,PROGRAM_CODE,,NaN,NaN,NaN
5442,REPORT_CATEGORY_LAB,,NaN,NaN,NaN


In [149]:
# merge the above merged df with output dataframe
df_final = pd.merge(df_output,df_merge,on='QUESTION_ID')
#df_final

In [150]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5444 entries, 0 to 5443
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   MODEL_NAME      5444 non-null   category
 1   QUESTION_ID     5444 non-null   object  
 2   QUESTIONSET_ID  5444 non-null   object  
 3   target_label    5444 non-null   object  
 4   Field           15 non-null     object  
 5   original_label  15 non-null     object  
 6   Label           15 non-null     object  
dtypes: category(1), object(6)
memory usage: 261.0+ KB


In [151]:
# Add, rename, remove columns as needed
df_final['target_table'] = ['' if f=='' else 'Custom__Treatment' for f in df_final.target_label]
df_final=df_final.rename(columns={'Field':'target_field'})
df_final=df_final.drop('Label',axis=1)


In [152]:
df_final = df_final[["MODEL_NAME",'QUESTION_ID','target_table','target_field','target_label']]
#df_final

In [153]:
df_final

,MODEL_NAME,QUESTION_ID,target_table,target_field,target_label
0,DiseaseSurveillanceModel_General,CURRENT_TELEPHONE_WORK,,NaN,
1,DiseaseSurveillanceModel_General,EAT_YOGURT_PURCHASED_FROM,,NaN,
2,DiseaseSurveillanceModel_General,CDR_PERSON_OR_AGENCY_REPORTING,,NaN,
3,DiseaseSurveillanceModel_General,ANTACID_EXPOSURE,,NaN,
4,DiseaseSurveillanceModel_General,DIARY_DAY_1_DINNER,,NaN,
...,...,...,...,...,...
5439,DiseaseSurveillanceModel_AggNETSS,COMMENTS,,NaN,
5440,IsolationAndControlModel,PRODUCT_CODE,,NaN,
5441,IsolationAndControlModel,PROGRAM_CODE,,NaN,
5442,IsolationAndControlModel,REPORT_CATEGORY_LAB,,NaN,


In [154]:
df_final.loc[:,'MODEL_NAME'].value_counts(dropna=False)

MODEL_NAME
DiseaseSurveillanceModel_General        3192
DiseaseSurveillanceModel_STD            1221
Childhood_Lead_Child_Model               361
DiseaseSurveillanceModel_Hep             296
ClusterModel                             195
Childhood_Lead_Investigation_Model       125
PortalApplicationModel                    30
DiseaseSurveillanceModel_Coinfection      10
EHRAndScreeningModel_STD                   8
IsolationAndControlModel                   4
DiseaseSurveillanceModel_AggNETSS          2
Name: count, dtype: int64

In [155]:
# Convert category type back to str object
df_final.MODEL_NAME = df_final.MODEL_NAME.astype('str')

In [156]:
# Rename model names
df_final.loc[:,'MODEL_NAME'] = [ m.split('_')[1] if 'DiseaseSurveillanceModel_' in m else m for m in df_final.loc[:,'MODEL_NAME']]

In [157]:
df_final.loc[:,'MODEL_NAME'].value_counts(dropna=False)

MODEL_NAME
General                               3192
STD                                   1221
Childhood_Lead_Child_Model             361
Hep                                    296
ClusterModel                           195
Childhood_Lead_Investigation_Model     125
PortalApplicationModel                  30
Coinfection                             10
EHRAndScreeningModel_STD                 8
IsolationAndControlModel                 4
AggNETSS                                 2
Name: count, dtype: int64

In [158]:
#df_final[df_final['target_field'].isna()]

In [159]:
df_final.to_csv('output/maven_Treatment_origami_similarity_match.csv',index=False)